# Advanced investigation: Company size, exposure and insurance claim frequency

**MATH4SDG / UDSM — optional advanced companion.** Start with the [introductory insurance notebook](synthetic_insurance_claims.ipynb) if you are new to pandas and Poisson models. This investigation is independent: you can run its cells without executing the introductory notebook first.

> **ALL DATA ARE SYNTHETIC.** The establishments, claims and rates here were invented for teaching. They are not records or statistics from WCF, TIRA, any insurer or Tanzania. This is an optional investigation, not a required industry-project activity.

**Questions.** Why do larger companies tend to have more claims? Why does plotting observed claim frequency against exposure reveal hyperbolas? Is a company's workforce size associated with its underlying claim rate after exposure is accounted for?

We distinguish **workforce size** (covered workers), **exposure** (worker-years), **claim count** (an integer) and **observed claim rate** (claims per worker-year). These are not interchangeable.


## 1. Generate and inspect one synthetic year

We use **2024** for a first investigation. Each fictional establishment appears once, so we are not accidentally treating five years of observations from the same establishment as five independent companies. The 2025 rate shift in the generator is also excluded. The data generator is deterministic and requires no internet access.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

import bootstrap
from tools.synthetic_insurance import generate, SYNTHETIC_MARKER


In [ ]:
data = generate()
assert (data["dataset_status"] == SYNTHETIC_MARKER).all()
sample = data.loc[data["year"] == 2024].copy()
sample["sector"] = sample["sector"].fillna("Unreported")
sample["claims_per_100"] = (
    100 * sample["claims_count"] / sample["worker_years"]
)
sample["log_workers"] = np.log(sample["workers"])
sample["log_exposure"] = np.log(sample["worker_years"])

print(f"{len(sample)} invented establishments in 2024")
sample[["workers", "worker_years", "claims_count", "claims_per_100"]].head()


## 2. How are raw claim counts related to workforce size?

Every claim is counted once, but companies differ in the number of workers they cover. Start with a plot using workers as a measure of **company size**. Does a larger count establish a higher *risk per worker*? Why not?


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(
    sample["workers"], sample["claims_count"],
    alpha=0.35, s=15,
)
ax.set_xlabel("Company size (covered workers)")
ax.set_ylabel("Number of synthetic claims in 2024")
ax.set_title("Synthetic claim counts and company size")


Actual exposure also depends on how much of the year the workers were covered. Under a **constant claim rate** $\lambda$, the simplest model is

$$
N_i \mid e_i\sim\operatorname{Poisson}(\lambda e_i),
\qquad
\mathbb E[N_i\mid e_i]=\lambda e_i.
$$

Here $e_i$ is worker-year exposure. Fit the descriptive pooled rate for 2024, then add the corresponding line to a count-versus-exposure scatter plot. This fitted line is **not** an independently validated forecast.


In [ ]:
pooled_rate = sample["claims_count"].sum() / sample["worker_years"].sum()
exposure_grid = np.linspace(0, sample["worker_years"].max(), 200)

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(
    sample["worker_years"], sample["claims_count"],
    alpha=0.30, s=15, label="Synthetic establishment-years",
)
ax.plot(
    exposure_grid, pooled_rate * exposure_grid,
    label="Constant-rate baseline", linewidth=2,
)
ax.set_xlabel("Exposure (worker-years)")
ax.set_ylabel("Number of synthetic claims in 2024")
ax.set_title("Expected counts increase with exposure")
ax.legend()


## 3. Why are observed frequencies arranged on curves?

Define the **observed** frequency per 100 worker-years by

$$
R_i=\frac{100N_i}{e_i}.
$$

Claim count $N_i$ must be a nonnegative integer. Thus, if $N_i=n$, its point in an exposure–rate plot lies on the hyperbola

$$
R=\frac{100n}{e},\qquad n=0,1,2,\ldots
$$

The horizontal row at zero corresponds to $n=0$. The bands are a mathematical consequence of discrete counts divided by exposure; they are **not** evidence of separate company-risk populations.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(
    sample["worker_years"], sample["claims_per_100"],
    alpha=0.30, s=15, label="Synthetic observed rates",
)
curve_exposure = np.linspace(
    max(25.0, float(sample["worker_years"].min())),
    float(sample["worker_years"].max()), 300,
)
for n in (1, 2, 3, 5):
    ax.plot(
        curve_exposure, 100 * n / curve_exposure,
        linewidth=1.1, label=f"{n} claim(s)",
    )
ax.set_xlabel("Exposure (worker-years)")
ax.set_ylabel("Claims per 100 worker-years")
ax.set_title("Integer counts create hyperbolas in observed rates")
ax.legend(ncol=3, fontsize=8)


Even **if all establishments had exactly the same underlying rate** $\lambda$, the conditional model above gives

$$
\mathbb E[R_i\mid e_i]=100\lambda,\qquad
\operatorname{Var}(R_i\mid e_i)=\frac{100^2\lambda}{e_i}.
$$

Thus, the observed rates of smaller-exposure establishments vary much more. In our actual synthetic generator, **underlying rates also differ** by sector, zone, programme and an unobserved establishment factor. The constant-rate formulas are a model to understand, not a complete description of the simulated data.

**Questions:** What happens to the spacing between adjacent hyperbolas when exposure doubles? Why is a single claim a much larger *rate* for a small company?


## 4. Aggregate claim frequency by company size

Grouping by workforce size gives more stable *descriptive* rates than averaging individual observed rates. The pooled rate within each group must be

$$
\widehat\lambda_{\mathrm{group}}
=\frac{\sum_{i\in\mathrm{group}}N_i}
{\sum_{i\in\mathrm{group}}e_i}.
$$

Use five approximate workforce-size groups (quintiles). Do **not** compute the unweighted mean of individual rates. Workers define the groups; worker-years supply the denominator.


In [ ]:
sample["size_group"] = pd.qcut(
    sample["workers"], q=5, duplicates="drop"
)
by_size = sample.groupby("size_group", observed=True).agg(
    establishments=("establishment_id", "nunique"),
    claims=("claims_count", "sum"),
    worker_years=("worker_years", "sum"),
)
by_size["claims_per_100_worker_years"] = (
    100 * by_size["claims"] / by_size["worker_years"]
)
by_size.round(2)


In [ ]:
ax = by_size["claims_per_100_worker_years"].plot.barh(
    figsize=(9, 4),
    title="SYNTHETIC claim frequency by workforce size (2024)",
)
ax.set_xlabel("Claims per 100 worker-years")
ax.set_ylabel("Company-size group (covered workers)")


**Interpretation challenge.** Are the group rates similar? If they differ, is it necessarily because workforce size changes underlying risk? Consider random sampling variation, differing exposures and the possible distribution of sectors across size groups. In real data, company size could genuinely matter; this particular **synthetic generator does not insert a direct size coefficient** in its claim-rate formula.


## 5. Advanced: a Poisson model with a size term and an exposure offset

To distinguish **expected counts** from **claim rates**, consider

$$
N_i\mid e_i,w_i,x_i \sim \operatorname{Poisson}(\mu_i),\qquad
\log\mu_i=\log e_i+\beta_0+\beta_1\log w_i+x_i^\top\gamma.
$$

Here $w_i$ denotes workers and $\log e_i$ is an **offset**, with coefficient fixed at 1. Without the offset, an apparent size effect may merely reflect that larger companies expose more worker-years.

We compare a size-only rate model with one also adjusting for the recorded sector, zone and programme. These are *associational* models, not causal estimates. We use a heteroscedasticity-robust covariance estimate (HC3) as a partial safeguard against variance misspecification; it does not establish that the Poisson mean model or the covariates are adequate.


In [ ]:
model_size = smf.glm(
    "claims_count ~ log_workers",
    data=sample,
    family=sm.families.Poisson(),
    offset=sample["log_exposure"],
).fit(cov_type="HC3")

model_adjusted = smf.glm(
    "claims_count ~ log_workers + C(sector) + C(zone) + C(programme)",
    data=sample,
    family=sm.families.Poisson(),
    offset=sample["log_exposure"],
).fit(cov_type="HC3")


If $\beta_1$ is the coefficient of $\log w$, **doubling the number of workers** corresponds to a rate ratio of

$$
\exp(\beta_1\log 2)=2^{\beta_1},
$$

at fixed other model covariates. A ratio near 1 suggests little estimated size association under the model. A confidence interval describes estimation uncertainty *conditional on the modelling assumptions*, not certainty about the true data-generating mechanism.


In [ ]:
def doubling_summary(fitted, description):
    beta = fitted.params["log_workers"]
    lower, upper = fitted.conf_int().loc["log_workers"]
    return {
        "model": description,
        "coefficient": beta,
        "rate_ratio_when_workers_double": np.exp(beta * np.log(2)),
        "interval_lower": np.exp(lower * np.log(2)),
        "interval_upper": np.exp(upper * np.log(2)),
    }

pd.DataFrame([
    doubling_summary(model_size, "Size only"),
    doubling_summary(model_adjusted, "Adjusted for sector, zone, programme"),
]).round(3)


The original generator uses exposure to scale the expected number of claims and has **no explicit workforce-size term** in the underlying frequency. This does not guarantee an estimated coefficient of exactly zero in a finite sample. Interpret a difference between the two fitted size coefficients cautiously; omitted factors, finite-sample noise and differences in assumptions may matter.

The generator includes unobserved establishment-level risk, so the Poisson variance assumption is not exactly correct after averaging over that heterogeneity. The following ratio gives a basic indication of whether residual variation exceeds the fitted Poisson variance; it is **not** a complete diagnostic.


In [ ]:
pearson_dispersion = (
    np.sum(model_adjusted.resid_pearson**2)
    / model_adjusted.df_resid
)
print(f"Pearson dispersion estimate: {pearson_dispersion:.2f}")


## 6. Further questions for independent investigation

1. Recompute the descriptive size-group rates for 2021, 2022, 2023 and 2025 separately. Are apparent patterns stable? The generator introduces a baseline change in 2025.
2. Do the rate ratios change if workforce groups are defined differently? Try three groups instead of five.
3. What happens if you remove sector from the adjusted model? Does the interpretation change?
4. What assumptions about risk and exposure would be required to use these models with genuine insurance data?
5. How would you test whether the observed size relationship is predictive for a **future year**, rather than merely describing the same year used to fit the model?

**Report:** Give one plot, one mathematical explanation, one modelling comparison and at least one important limitation. Do not report synthetic estimates as real insurance statistics. Industry-provided data may have confidentiality restrictions and must not be uploaded to public repositories or external AI services without permission.
